# Etude de la mortalité en Île-de-France

Nous souhaitons lier la pollution à certains indices, comme la mortalité, pour tenter d'observer si corrélation il y a, voire aller plus loin jusqu'à étudier une causalité. 

Pour ce faire, nous récupérons des données sur le nombre de décès par commune en France sur datagouv. Nous allons extraire ces données et les restreindre à l'Île-de-France pour en faire une première analyse. Il est important de noter que la variable géographique liée au décès est le code_commune du décès : c'est à dire l'endroit où la personne est décédée. Ainsi, il n'indique peut être pas correctement le lieu où la personne vivait et était donc soumise à la pollution atmosphérique. C'est un détail à garder en tête pour la suite de notre analyse. 

Afin d'obtenir un indicateur un peu plus performant que simplement le nombre de décès (qui peut être trompeur du fait de l'hétérogénéité de la densité de la population, mais qui ne témoigne également pas de potentiels mouvements de population, d'exode rural ou bien de désertification), nous récupérons également des données sur le nombre d'habitants par commune. Ainsi, nous pourrons calculer un taux de mortalité - imparfait du fait de la limite mentionnée précédemment - qui nous permettra une analyse plus fine. 

In [ ]:
# Téléchargement des packages

import pandas as pd
import requests
import io
import zipfile

Nous commençons par télécharger les données sur le nombre de décès à partir de data.gouv.

In [2]:
url_deces = f"https://www.data.gouv.fr/api/1/datasets/r/0b0c0fbc-af9a-4b9d-9949-b171e7e6fe4a"
deces = pd.read_csv(url_deces, sep=';', dtype=str)
deces.head()

,code_lieu_deces,deces_2008,deces_2009,deces_2010,deces_2011,deces_2012,deces_2013,deces_2014,deces_2015,deces_2016,...,deces_2024,deces_2025,total_2008_2025,moy_annuelle,annee_pic,deces_annee_pic,evol_pct_2008_vs_2025,tendance,surplus_covid_2020,surplus_pct_covid_2020
0,31555,4559,4647,4650,4763,4924,4877,4981,5208,5153,...,5433,5626,92323,5129.1,2022,5642,23.4,Forte hausse,-9,-0.2
1,06088,4663,4576,4323,4416,4703,4763,4667,4803,4957,...,4651,4695,85661,4758.9,2021,5253,0.7,Stable,163,3.4
2,59350,4155,4224,4182,4081,4346,4499,4469,4466,4586,...,4334,4436,80527,4473.7,2020,4886,6.8,Hausse,225,4.8
3,33063,4155,4212,4178,4090,4306,4236,4379,4535,4221,...,4142,4108,76144,4230.2,2015,4535,-1.1,Stable,-124,-2.9
4,67482,3717,3743,3765,3753,3893,4188,4045,4213,4016,...,4380,4415,74441,4135.6,2020,4730,18.8,Hausse,619,15.1


In [13]:
deces.info()

<class 'pandas.DataFrame'>
RangeIndex: 36765 entries, 0 to 36764
Data columns (total 28 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   code_commune            36765 non-null  str  
 1   deces_2008              36765 non-null  str  
 2   deces_2009              36765 non-null  str  
 3   deces_2010              36765 non-null  str  
 4   deces_2011              36765 non-null  str  
 5   deces_2012              36765 non-null  str  
 6   deces_2013              36765 non-null  str  
 7   deces_2014              36765 non-null  str  
 8   deces_2015              36765 non-null  str  
 9   deces_2016              36765 non-null  str  
 10  deces_2017              36765 non-null  str  
 11  deces_2018              36765 non-null  str  
 12  deces_2019              36765 non-null  str  
 13  deces_2020              36765 non-null  str  
 14  deces_2021              36765 non-null  str  
 15  deces_2022              36765 

Nous avons donc bien le code_lieu_deces, qui correspond au code_commune INSEE, ainsi que le nombre de décès au sein de la commune, de l'année 2008 à 2025. Certaines statistiques descriptives sont déjà calculées, comme le total sur cette période, la moyenne annuelle, l'année pic ou bie, l'évolution entre 2008 et 2025. Une certaine importance est déjà donnée au Covid et à raison : la pandémie a à la fois augmenter le nombre de décès, mais le confinement a pu réduire la pollution atmosphérique ce qui rend l'effet global sur la mortalité ambigu. 

A présent, nous nous concentrons sur les données en Île de France. Pour cela nous allons créer une colonne département à partir de code_lieu_décès qui correspond au code de la commune. Les deux premiers chiffres sont isolés, et nous nous limitons ensuite au département d'Île-de-France. 

In [19]:
deces = deces.rename({"code_lieu_deces": "code_commune"}, axis=1)

deces["dep"] = deces["code_commune"].str[:2]
deces[["dep", "code_commune"]]
#rq : 99 fait référence aux pays étrangers

,dep,code_commune
0,31,31555
1,06,06088
2,59,59350
3,33,33063
4,67,67482
...,...,...
36760,11,11094
36761,77,77356
36762,88,88117
36763,80,80035


In [16]:
deces["dep"].value_counts().sort_index()

dep
01    417
02    804
03    320
04    198
05    175
     ... 
94     47
95    183
97    136
98     82
99    198
Name: count, Length: 100, dtype: int64

Pour chaque commune, le département est bien affiché. Le nombre de commune par département semble également cohérent. 

Pour l'Ain (01), il existe 419 communes et nous en avons 417. Pour l'Aisne, 804 pour 816, etc. 

Par ailleurs, aucune commune en Île-de-France n'a changé de département depuis 2008, donc il ne devrait pas y avoir de problème quant à cela. 

In [ ]:
dep_idf = ["75", "77", "78", "91", "92", "93", "94", "95"]
deces_idf = deces[deces["dep"].isin(dep_idf)].copy()

#nombre de communes par département 
print(deces_idf["dep"].value_counts().sort_index())

dep
75     20
77    513
78    262
91    193
92     36
93     41
94     47
95    183
Name: count, dtype: int64


Encore une fois le nombre de commune par département semble cohérent : nous avons les 20 départements de Paris, exactement le bon nombre de commune pour le 92, 94, 95, et un nombre à quelques unités proches pour les autres. Cela peut s'expliquer par la fusion ou la séparation de commune. Nous gardons cela en tête et verrons par la suite si cela nous pose un problème, mais les données semblent donc très complète. 

Il est à présent possible d'observer quelles sont les communes avec le plus grand nombre de décès. 

Afin de calculer un indice de taux de mortalité : il faut également avoir en tête que chaque commune ne compte pas le même nombre d'habitants. Ainsi, étudier l'évolution de la mortalité passe par étudier l'évolution de la démographie. 

Pour cela, nous téléchargons des données concernant le nombre d'habitants. Un des fichiers les mieux renseignés est celui sur les populations municipales (donc par communes : la différence est très minime et repose sur l'inclusion des personnes ayant plusieurs lieux de résidence). 

In [6]:
url_pop = f"https://www.data.gouv.fr/api/1/datasets/r/27361ae9-dcab-4312-97a1-48dab5f39218"
r = requests.get(url_pop)

In [7]:
z = zipfile.ZipFile(io.BytesIO(r.content))
print(z.namelist())   # les fichiers du zip

['DS_POPULATIONS_HISTORIQUES_metadata.csv', 'DS_POPULATIONS_HISTORIQUES_data.csv']


In [8]:
#le fichier que nous souhaitons avoir est celui qui ne contient pas les metadata
fichier_data_pop = [n for n in z.namelist() if "metadata" not in n][0]

#on ouvre le zip
with z.open(fichier_data_pop) as f:
    pop = pd.read_csv(f, sep=";", dtype=str)

print(pop.shape)
print(pop.columns.tolist())
pop.head()

(813234, 6)
['FREQ', 'GEO', 'GEO_OBJECT', 'POPREF_MEASURE', 'TIME_PERIOD', 'OBS_VALUE']


,FREQ,GEO,GEO_OBJECT,POPREF_MEASURE,TIME_PERIOD,OBS_VALUE
0,A,15113,COM,PMUN,2010,197
1,A,14491,COM,PMUN,2007,154
2,A,14761,COM,PMUN,2011,746
3,A,14700,COM,PMUN,2018,663
4,A,15038,COM,PMUN,2013,1040


Nous allons conserver les geo_object étant des communes, soit COM, et ensuite se concentrer sur l'Île de France. 
Mais tout d'abord, il est possible d'observer dès les premières lignes que le tableau est dans un format particulier : en effet, une ligne correspond à une année observée. La même commune est donc présente plusieurs fois. 

Nous modifions le format du tableau afin d'obtenir une commune = une ligne. 

Pour cela, nous gardons GEO_OBJECT = COM, les communes, et une mesure par population municipale, POPREF_MEASURE = PMUN.

In [9]:
pop_com = pop[(pop["POPREF_MEASURE"] == "PMUN") & (pop["GEO_OBJECT"] == "COM")].copy()

pop_com["GEO"] = pop_com["GEO"].astype(str).str.zfill(5)
pop_com["TIME_PERIOD"] = pop_com["TIME_PERIOD"].astype(int)
pop_com["OBS_VALUE"] = pd.to_numeric(pop_com["OBS_VALUE"], errors="coerce")

# on ne garde que la période utile par rapport aux données sur la mortalité 
pop_com = pop_com[pop_com["TIME_PERIOD"] >= 2008]

pop_wide = pop_com.pivot(index="GEO", columns="TIME_PERIOD", values="OBS_VALUE")
pop_wide.columns = [f"pop_{annee}" for annee in pop_wide.columns]
pop_wide = pop_wide.reset_index().rename(columns={"GEO": "code_commune"})

pop_wide.head()

,code_commune,pop_2008,pop_2009,pop_2010,pop_2011,pop_2012,pop_2013,pop_2014,pop_2015,pop_2016,pop_2017,pop_2018,pop_2019,pop_2020,pop_2021,pop_2022,pop_2023
0,01001,791,787,784,780,777,767,767,767,767,776,771,779,806,832,859,860
1,01002,194,207,221,234,235,236,239,241,243,248,253,256,262,267,273,270
2,01004,12792,13350,13835,13839,14233,14359,14022,14127,14081,14035,14204,14134,14288,14854,15554,15934
3,01005,1567,1592,1616,1600,1642,1635,1627,1619,1671,1689,1720,1751,1782,1897,1917,1906
4,01006,123,120,116,112,110,108,109,109,110,111,112,112,113,113,114,115


In [10]:
pop_wide.shape

(34858, 17)

In [11]:
pop_wide['dep'] = pop_wide['code_commune'].str[:2]
pop_idf = pop_wide[pop_wide["dep"].isin(dep_idf)].copy()
print(pop_idf["dep"].value_counts().sort_index())


dep
75      1
77    507
78    259
91    194
92     36
93     39
94     47
95    183
Name: count, dtype: int64


Il semble y avoir un problème avec paris : les arrondissements ont été absorbés. Ils correspondent en fait à un code particulier, ARM. Il faut adapter le code. 

In [12]:
pop_test = pop_wide[pop_wide['dep']=="75"]
pop_test

,code_commune,pop_2008,pop_2009,pop_2010,pop_2011,pop_2012,pop_2013,pop_2014,pop_2015,pop_2016,pop_2017,pop_2018,pop_2019,pop_2020,pop_2021,pop_2022,pop_2023,dep
29194,75056,2211297,2234105,2243833,2249975,2240621,2229621,2220445,2206488,2190327,2187526,2175601,2165423,2145906,2133111,2113705,2103778,75
